# AI-Based Loan Approval Prediction and Decision Support System

This Jupyter Notebook demonstrates the step-by-step machine learning workflow for predicting loan approval eligibility based on applicant demographics and financial features.

### Technologies Used:
- **Pandas & NumPy** for data manipulation
- **Matplotlib & Seaborn** for exploratory data visualization
- **Scikit-Learn** for building the preprocessing pipeline and training classifiers
- **Joblib** for model serialization

## 1. Import Libraries

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Set style for plots
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Load and Inspect Dataset

In [ ]:
dataset_path = '../dataset/loan_data.csv'
if not os.path.exists(dataset_path):
    # Attempt to load from current path if executed in notebook folder directly
    dataset_path = 'dataset/loan_data.csv'

df = pd.read_csv(dataset_path)
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
# Information on columns, data types and non-null values
df.info()

In [ ]:
# Missing values summary
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# Target distribution
print("Target Variable Distribution:")
print(df['Loan_Status'].value_counts(normalize=True))

## 3. Exploratory Data Analysis

In [ ]:
# Loan approval distribution
plt.figure()
status_counts = df['Loan_Status'].value_counts()
plt.pie(status_counts, labels=['Approved (Y)', 'Rejected (N)'], autopct='%1.1f%%', 
        colors=['#059669', '#dc2626'], startangle=90, explode=(0.05, 0))
plt.title('Overall Loan Status Approval Rates')
plt.show()

In [ ]:
# Credit History vs Loan Status
plt.figure()
sns.countplot(x='Credit_History', hue='Loan_Status', data=df.dropna(subset=['Credit_History']), palette=['#059669', '#dc2626'])
plt.title('Loan Approval Status by Credit History')
plt.xlabel('Credit History (1.0 = Guidelines Met, 0.0 = Guidelines Not Met)')
plt.ylabel('Count')
plt.show()

In [ ]:
# Education vs Loan Status
plt.figure()
sns.countplot(x='Education', hue='Loan_Status', data=df, palette=['#059669', '#dc2626'])
plt.title('Loan Approval Status by Education Level')
plt.xlabel('Education')
plt.ylabel('Count')
plt.show()

## 4. Feature Engineering

In [ ]:
# 1. Combine Applicant and Coapplicant incomes into TotalIncome
df['TotalIncome'] = df['ApplicantIncome'] + df['CoapplicantIncome']

# 2. Calculate Income-to-Loan Ratio (adding small epsilon to prevent division by zero)
df['Income_to_Loan_Ratio'] = df['TotalIncome'] / (df['LoanAmount'] + 0.001)

df[['ApplicantIncome', 'CoapplicantIncome', 'TotalIncome', 'LoanAmount', 'Income_to_Loan_Ratio']].head()

## 5. Build Preprocessing Pipeline & Splitting

In [ ]:
# Split features and target
X = df.drop(columns=['Loan_ID', 'Loan_Status'])
y = df['Loan_Status'].map({'Y': 1, 'N': 0})

# Columns list
numeric_features = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'TotalIncome', 'Income_to_Loan_Ratio']
categorical_features = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Credit_History', 'Property_Area']

# Set objects
for col in categorical_features:
    X[col] = X[col].astype(object)

# Pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Split dataset (Stratified to balance targets)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Fit preprocessor
preprocessor.fit(X_train)
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print("Transformed Training Shape:", X_train_transformed.shape)
print("Transformed Testing Shape:", X_test_transformed.shape)

## 6. Train and Compare Classifiers

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Support Vector Machine': SVC(kernel='rbf', probability=True, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42)
}

metrics_summary = {}

for name, model in models.items():
    model.fit(X_train_transformed, y_train)
    y_pred = model.predict(X_test_transformed)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    metrics_summary[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    }

comparison_df = pd.DataFrame(metrics_summary).T
comparison_df

## 7. Select and Save Best Model

In [ ]:
best_model_name = comparison_df['F1-Score'].idxmax()
print(f"Best performing model: {best_model_name} with F1-Score of {comparison_df.loc[best_model_name, 'F1-Score']:.4f}")

# Save best model & preprocessor
joblib.dump(models[best_model_name], '../model/best_model.pkl')
joblib.dump(preprocessor, '../model/preprocessor.pkl')
print("Saved pickle assets to '../model/' folder.")